# Final Project PPDKM — Prediksi Customer Churn Pelanggan Telco Menggunakan CRISP-DM

**Nama/NIM:** Louis Natasha Voudy Nanlessy/L0223026 
**Mata Kuliah:** PPDKM / Kapita Selekta  
**Dataset:** Telco Customer Churn  
**Metode:** Data Mining Classification dengan pendekatan CRISP-DM  

Notebook ini dibuat untuk memenuhi komponen:
1. Business Understanding  
2. Data Understanding  
3. Data Preparation  
4. Modeling / Design Experiment  
5. Evaluation dan Analisis  
6. Deployment sederhana menggunakan Streamlit  
7. Conclusion

In [ ]:
# ==============================================
# 0. Import Library
# ==============================================

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

import joblib

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

## 1. Business Understanding

### Latar Belakang
Customer churn adalah kondisi ketika pelanggan berhenti menggunakan layanan perusahaan. Pada perusahaan telekomunikasi, churn dapat menyebabkan penurunan pendapatan karena perusahaan kehilangan pelanggan aktif. Oleh karena itu, perusahaan perlu mengetahui pelanggan mana yang berpotensi churn agar dapat membuat strategi retensi pelanggan.

### Tujuan Penelitian
Tujuan project ini adalah membangun model data mining untuk memprediksi apakah pelanggan akan churn atau tidak berdasarkan data layanan, kontrak, tagihan, dan profil pelanggan.

### Rumusan Masalah
1. Bagaimana karakteristik pelanggan yang cenderung churn?
2. Bagaimana proses preprocessing dataset Telco Customer Churn?
3. Algoritma data mining mana yang memberikan performa terbaik?
4. Bagaimana hasil evaluasi model berdasarkan accuracy, precision, recall, F1-score, dan ROC-AUC?

### Batasan Masalah
1. Dataset yang digunakan adalah Telco Customer Churn.
2. Target prediksi adalah kolom `Churn`.
3. Algoritma yang digunakan adalah Logistic Regression, KNN, Decision Tree, Random Forest, Gradient Boosting, dan AdaBoost.
4. Evaluasi dilakukan menggunakan data testing dengan split 80:20.

## 2. Data Understanding

Tahap ini digunakan untuk memahami struktur dataset, jumlah data, tipe data, missing value, distribusi target, dan gambaran awal hubungan antar fitur.

In [ ]:
# ==============================================
# 2.1 Load Dataset
# ==============================================

possible_paths = [
    Path("Telco-Customer-Churn.csv"),
    Path("/mnt/data/Telco-Customer-Churn.csv"),
    Path("/kaggle/input/telco-customer-churn/Telco-Customer-Churn.csv"),
    Path("/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"),
]

DATA_PATH = None
for p in possible_paths:
    if p.exists():
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "File dataset tidak ditemukan. Pastikan file Telco-Customer-Churn.csv "
        "berada satu folder dengan notebook, atau ubah variabel DATA_PATH secara manual."
    )

df = pd.read_csv(DATA_PATH)
print(f"Dataset berhasil dibaca dari: {DATA_PATH}")
print(f"Jumlah baris dan kolom: {df.shape}")
df.head()

In [ ]:
# ==============================================
# 2.2 Informasi Struktur Dataset
# ==============================================

print("Ukuran dataset:", df.shape)
print("\nTipe data:")
display(df.dtypes)

print("\nStatistik deskriptif kolom numerik:")
display(df.describe())

print("\nContoh data:")
display(df.head())

In [ ]:
# ==============================================
# 2.3 Cek Missing Value, Duplikat, dan Nilai Unik
# ==============================================

missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().sum() / len(df) * 100).round(2),
    "unique_values": df.nunique()
}).sort_values("missing_count", ascending=False)

print("Jumlah data duplikat:", df.duplicated().sum())
display(missing_summary)

In [ ]:
# ==============================================
# 2.4 Cek Kolom TotalCharges
# TotalCharges sering terbaca sebagai object karena ada nilai kosong berupa spasi.
# ==============================================

print("Tipe data TotalCharges sebelum cleaning:", df["TotalCharges"].dtype)
print("Jumlah nilai TotalCharges kosong/spasi:", (df["TotalCharges"].astype(str).str.strip() == "").sum())

display(df[df["TotalCharges"].astype(str).str.strip() == ""].head())

In [ ]:
# ==============================================
# 2.5 Distribusi Target Churn
# ==============================================

target_counts = df["Churn"].value_counts()
target_percent = (df["Churn"].value_counts(normalize=True) * 100).round(2)

display(pd.DataFrame({
    "count": target_counts,
    "percent": target_percent
}))

plt.figure(figsize=(6, 4))
target_counts.plot(kind="bar")
plt.title("Distribusi Target Churn")
plt.xlabel("Churn")
plt.ylabel("Jumlah Pelanggan")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# ==============================================
# 2.6 Exploratory Data Analysis
# ==============================================

# Churn berdasarkan jenis kontrak
contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100
display(contract_churn.round(2))

plt.figure(figsize=(8, 4))
contract_churn["Yes"].sort_values(ascending=False).plot(kind="bar")
plt.title("Persentase Churn Berdasarkan Jenis Kontrak")
plt.xlabel("Contract")
plt.ylabel("Persentase Churn (%)")
plt.xticks(rotation=20)
plt.show()

# Churn berdasarkan InternetService
internet_churn = pd.crosstab(df["InternetService"], df["Churn"], normalize="index") * 100
display(internet_churn.round(2))

plt.figure(figsize=(8, 4))
internet_churn["Yes"].sort_values(ascending=False).plot(kind="bar")
plt.title("Persentase Churn Berdasarkan Internet Service")
plt.xlabel("Internet Service")
plt.ylabel("Persentase Churn (%)")
plt.xticks(rotation=20)
plt.show()

In [ ]:
# ==============================================
# 2.7 Distribusi Numerik Berdasarkan Churn
# ==============================================

# TotalCharges diubah sementara ke numerik untuk visualisasi
df_eda = df.copy()
df_eda["TotalCharges"] = pd.to_numeric(df_eda["TotalCharges"], errors="coerce")

numeric_cols_eda = ["tenure", "MonthlyCharges", "TotalCharges"]

for col in numeric_cols_eda:
    plt.figure(figsize=(7, 4))
    for churn_value in df_eda["Churn"].dropna().unique():
        subset = df_eda[df_eda["Churn"] == churn_value][col].dropna()
        plt.hist(subset, bins=30, alpha=0.5, label=f"Churn = {churn_value}")
    plt.title(f"Distribusi {col} Berdasarkan Churn")
    plt.xlabel(col)
    plt.ylabel("Frekuensi")
    plt.legend()
    plt.show()

## 3. Data Preparation

Tahap data preparation dilakukan untuk menyiapkan data sebelum masuk ke model. Proses yang dilakukan:
1. Mengubah `TotalCharges` dari object menjadi numeric.
2. Menghapus baris yang memiliki `TotalCharges` kosong.
3. Menghapus kolom `customerID` karena hanya identitas pelanggan dan tidak digunakan sebagai fitur prediksi.
4. Mengubah target `Churn` menjadi bentuk biner: `Yes = 1`, `No = 0`.
5. Memisahkan fitur numerik dan kategorikal.
6. Membuat preprocessing pipeline:
   - Numerik: imputasi median dan standardisasi.
   - Kategorikal: imputasi modus dan one-hot encoding.

In [ ]:
# ==============================================
# 3.1 Cleaning Data
# ==============================================

data = df.copy()

# Ubah TotalCharges menjadi numeric
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")

# Hapus baris TotalCharges kosong
before_rows = data.shape[0]
data = data.dropna(subset=["TotalCharges"]).reset_index(drop=True)
after_rows = data.shape[0]

print(f"Jumlah baris sebelum cleaning TotalCharges: {before_rows}")
print(f"Jumlah baris setelah cleaning TotalCharges: {after_rows}")
print(f"Jumlah baris terhapus: {before_rows - after_rows}")

# Hapus customerID
if "customerID" in data.columns:
    data = data.drop(columns=["customerID"])

# Encoding target
data["Churn"] = data["Churn"].map({"Yes": 1, "No": 0})

display(data.head())
print(data.info())

In [ ]:
# ==============================================
# 3.2 Pisahkan Fitur dan Target
# ==============================================

X = data.drop(columns=["Churn"])
y = data["Churn"]

numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Jumlah fitur numerik:", len(numerical_features))
print(numerical_features)

print("\nJumlah fitur kategorikal:", len(categorical_features))
print(categorical_features)

In [ ]:
# ==============================================
# 3.3 Train-Test Split
# ==============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Ukuran X_train:", X_train.shape)
print("Ukuran X_test:", X_test.shape)
print("Distribusi target train:")
display(y_train.value_counts(normalize=True).rename("proportion").to_frame())

print("Distribusi target test:")
display(y_test.value_counts(normalize=True).rename("proportion").to_frame())

In [ ]:
# ==============================================
# 3.4 Preprocessing Pipeline
# ==============================================

# Kompatibilitas untuk versi scikit-learn lama dan baru
try:
    onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", onehot_encoder)
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

preprocessor

## 4. Modeling / Design Experiment

### Design Experiment
Eksperimen dilakukan dengan membandingkan beberapa algoritma klasifikasi. Semua algoritma menggunakan data train-test split yang sama, yaitu 80% data training dan 20% data testing. Target yang diprediksi adalah `Churn`.

Algoritma yang digunakan:
1. Logistic Regression
2. K-Nearest Neighbor
3. Decision Tree
4. Random Forest
5. Gradient Boosting
6. AdaBoost

Metrik evaluasi:
- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC

Pada kasus churn, **F1-Score dan Recall** penting karena perusahaan perlu mendeteksi pelanggan yang berpotensi churn. Recall tinggi berarti model lebih mampu menangkap pelanggan yang benar-benar churn.

In [ ]:
# ==============================================
# 4.1 Definisi Model
# ==============================================

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "K-Nearest Neighbor": KNeighborsClassifier(
        n_neighbors=7
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    ),
    "AdaBoost": AdaBoostClassifier(
        random_state=RANDOM_STATE
    )
}

models

In [ ]:
# ==============================================
# 4.2 Training dan Evaluasi Semua Model
# ==============================================

results = []
trained_models = {}

for model_name, algorithm in models.items():
    print(f"Training model: {model_name}")

    model_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", algorithm)
    ])

    model_pipeline.fit(X_train, y_train)

    y_pred = model_pipeline.predict(X_test)
    y_proba = model_pipeline.predict_proba(X_test)[:, 1]

    result = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    }

    results.append(result)
    trained_models[model_name] = model_pipeline

result_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False).reset_index(drop=True)

print("Hasil perbandingan model:")
display(result_df)

In [ ]:
# ==============================================
# 4.3 Visualisasi Perbandingan Model
# ==============================================

metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]

for metric in metrics_to_plot:
    plt.figure(figsize=(9, 4))
    plot_data = result_df.sort_values(metric, ascending=True)
    plt.barh(plot_data["Model"], plot_data[metric])
    plt.title(f"Perbandingan Model Berdasarkan {metric}")
    plt.xlabel(metric)
    plt.xlim(0, 1)
    plt.show()

## 5. Evaluation dan Analisis

Model terbaik dipilih berdasarkan F1-Score karena dataset churn memiliki ketidakseimbangan kelas. Selain F1-Score, nilai recall juga diperhatikan karena model diharapkan mampu mendeteksi pelanggan yang berpotensi churn.

In [ ]:
# ==============================================
# 5.1 Pilih Model Terbaik
# ==============================================

best_model_name = result_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print(f"Model terbaik berdasarkan F1-Score: {best_model_name}")
display(result_df.iloc[[0]])

In [ ]:
# ==============================================
# 5.2 Confusion Matrix dan Classification Report
# ==============================================

y_pred_best = best_model.predict(X_test)
y_proba_best = best_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred_best)

print("Confusion Matrix:")
print(cm)

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks([0, 1], ["No Churn", "Churn"])
plt.yticks([0, 1], ["No Churn", "Churn"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=["No Churn", "Churn"]))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_best))

In [ ]:
# ==============================================
# 5.3 ROC Curve Semua Model
# ==============================================

plt.figure(figsize=(8, 6))

for model_name, model_pipeline in trained_models.items():
    y_proba = model_pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_score = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{model_name} (AUC = {auc_score:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random Guess")
plt.title("ROC Curve Model")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

In [ ]:
# ==============================================
# 5.4 Analisis Feature Importance dari Random Forest
# ==============================================

# Feature importance digunakan untuk memahami fitur apa yang paling berpengaruh.
# Karena pipeline menggunakan one-hot encoding, nama fitur perlu diambil dari preprocessing.

rf_model = trained_models["Random Forest"]
rf_preprocessor = rf_model.named_steps["preprocessor"]
rf_classifier = rf_model.named_steps["model"]

# Ambil nama fitur hasil transformasi
num_feature_names = numerical_features

cat_encoder = rf_preprocessor.named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = cat_encoder.get_feature_names_out(categorical_features).tolist()

all_feature_names = num_feature_names + cat_feature_names

importance_df = pd.DataFrame({
    "feature": all_feature_names,
    "importance": rf_classifier.feature_importances_
}).sort_values("importance", ascending=False)

print("Top 20 fitur paling penting berdasarkan Random Forest:")
display(importance_df.head(20))

plt.figure(figsize=(8, 6))
top_features = importance_df.head(15).sort_values("importance", ascending=True)
plt.barh(top_features["feature"], top_features["importance"])
plt.title("Top 15 Feature Importance - Random Forest")
plt.xlabel("Importance")
plt.show()

In [ ]:
# ==============================================
# 5.5 Interpretasi Hasil Otomatis
# ==============================================

best_row = result_df.iloc[0]

print("INTERPRETASI HASIL")
print("=" * 80)
print(f"Model terbaik pada eksperimen ini adalah {best_row['Model']}.")
print(f"Accuracy  : {best_row['Accuracy']:.4f}")
print(f"Precision : {best_row['Precision']:.4f}")
print(f"Recall    : {best_row['Recall']:.4f}")
print(f"F1-Score  : {best_row['F1-Score']:.4f}")
print(f"ROC-AUC   : {best_row['ROC-AUC']:.4f}")
print()
print("Analisis:")
print("- Accuracy menunjukkan proporsi prediksi benar dari seluruh data testing.")
print("- Precision menunjukkan ketepatan model saat memprediksi pelanggan churn.")
print("- Recall menunjukkan kemampuan model menangkap pelanggan yang benar-benar churn.")
print("- F1-Score menyeimbangkan precision dan recall, sehingga cocok untuk kasus churn.")
print("- ROC-AUC menunjukkan kemampuan model membedakan kelas churn dan tidak churn.")
print()
print("Pada kasus customer churn, recall dan F1-Score penting karena perusahaan lebih baik")
print("mengetahui pelanggan yang berisiko churn lebih awal agar dapat diberikan strategi retensi.")

## 6. Deployment

Pada tahap deployment, model terbaik disimpan dalam format `.pkl` menggunakan `joblib`. Notebook ini juga membuat file `streamlit_app.py` sebagai contoh aplikasi sederhana untuk prediksi churn.

Cara menjalankan deployment setelah semua cell dijalankan:
```bash
streamlit run streamlit_app.py
```

In [ ]:
# ==============================================
# 6.1 Simpan Model Terbaik dan Schema Input
# ==============================================

model_filename = "telco_churn_best_model.pkl"
schema_filename = "feature_schema.json"

joblib.dump(best_model, model_filename)

feature_schema = {
    "best_model_name": best_model_name,
    "numerical_features": numerical_features,
    "categorical_features": categorical_features,
    "categorical_options": {
        col: sorted([str(x) for x in X[col].dropna().unique().tolist()])
        for col in categorical_features
    },
    "numerical_defaults": {
        col: float(X[col].median())
        for col in numerical_features
    },
    "numerical_min": {
        col: float(X[col].min())
        for col in numerical_features
    },
    "numerical_max": {
        col: float(X[col].max())
        for col in numerical_features
    }
}

with open(schema_filename, "w") as f:
    json.dump(feature_schema, f, indent=4)

print(f"Model tersimpan sebagai: {model_filename}")
print(f"Schema input tersimpan sebagai: {schema_filename}")

In [ ]:
# ==============================================
# 6.2 Buat File Deployment Streamlit
# ==============================================

streamlit_code = '''
import json
import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Telco Customer Churn Prediction", layout="wide")

st.title("Prediksi Customer Churn Pelanggan Telco")
st.write("Aplikasi sederhana untuk memprediksi apakah pelanggan berpotensi churn atau tidak.")

MODEL_PATH = "telco_churn_best_model.pkl"
SCHEMA_PATH = "feature_schema.json"

model = joblib.load(MODEL_PATH)

with open(SCHEMA_PATH, "r") as f:
    schema = json.load(f)

st.subheader("Input Data Pelanggan")

input_data = {}

col1, col2 = st.columns(2)

with col1:
    st.markdown("### Fitur Numerik")
    for col in schema["numerical_features"]:
        min_value = schema["numerical_min"][col]
        max_value = schema["numerical_max"][col]
        default_value = schema["numerical_defaults"][col]

        if col == "SeniorCitizen":
            input_data[col] = st.selectbox(
                col,
                options=[0, 1],
                format_func=lambda x: "Yes" if x == 1 else "No"
            )
        else:
            input_data[col] = st.number_input(
                col,
                min_value=float(min_value),
                max_value=float(max_value),
                value=float(default_value)
            )

with col2:
    st.markdown("### Fitur Kategorikal")
    for col in schema["categorical_features"]:
        options = schema["categorical_options"][col]
        input_data[col] = st.selectbox(col, options=options)

if st.button("Prediksi Churn"):
    input_df = pd.DataFrame([input_data])

    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0][1]

    st.subheader("Hasil Prediksi")

    if prediction == 1:
        st.error("Pelanggan diprediksi CHURN")
    else:
        st.success("Pelanggan diprediksi TIDAK CHURN")

    st.write(f"Probabilitas Churn: {probability:.2%}")

    st.markdown("### Interpretasi")
    if probability >= 0.7:
        st.write("Risiko churn tinggi. Pelanggan perlu menjadi prioritas retensi.")
    elif probability >= 0.4:
        st.write("Risiko churn sedang. Pelanggan perlu dipantau dan dapat diberikan promo/penawaran.")
    else:
        st.write("Risiko churn rendah. Pelanggan relatif aman.")
'''

with open("streamlit_app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_code)

print("File streamlit_app.py berhasil dibuat.")
print("Jalankan dengan perintah: streamlit run streamlit_app.py")

## 7. Optional: Cross Validation

Cell di bawah bersifat opsional. Jika ingin hasil evaluasi lebih kuat, ubah `RUN_CV = True`.  
Kalau laptop lambat, biarkan `False` saja karena training beberapa algoritma dengan cross validation bisa memakan waktu lebih lama.

In [ ]:
# ==============================================
# 7. Optional Cross Validation
# ==============================================

RUN_CV = False

if RUN_CV:
    from sklearn.model_selection import StratifiedKFold, cross_validate

    scoring = {
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    cv_results = []

    for model_name, algorithm in models.items():
        print(f"Cross validation model: {model_name}")

        model_pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", algorithm)
        ])

        scores = cross_validate(
            model_pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=1
        )

        cv_results.append({
            "Model": model_name,
            "CV Accuracy": scores["test_accuracy"].mean(),
            "CV Precision": scores["test_precision"].mean(),
            "CV Recall": scores["test_recall"].mean(),
            "CV F1-Score": scores["test_f1"].mean(),
            "CV ROC-AUC": scores["test_roc_auc"].mean()
        })

    cv_result_df = pd.DataFrame(cv_results).sort_values("CV F1-Score", ascending=False)
    display(cv_result_df)
else:
    print("Cross validation tidak dijalankan. Ubah RUN_CV = True jika ingin menjalankan.")

## 8. Conclusion

Berdasarkan eksperimen yang dilakukan, model data mining berhasil dibangun untuk memprediksi customer churn menggunakan dataset Telco Customer Churn. Proses penelitian mengikuti pendekatan CRISP-DM, mulai dari business understanding, data understanding, data preparation, modeling, evaluation, hingga deployment sederhana.

Kesimpulan yang dapat ditulis pada laporan:
1. Dataset Telco Customer Churn memiliki target berupa status churn pelanggan.
2. Preprocessing penting dilakukan karena kolom `TotalCharges` awalnya bertipe object dan memiliki nilai kosong berupa spasi.
3. Fitur kategorikal diolah menggunakan one-hot encoding, sedangkan fitur numerik distandardisasi.
4. Beberapa algoritma berhasil dibandingkan, yaitu Logistic Regression, KNN, Decision Tree, Random Forest, Gradient Boosting, dan AdaBoost.
5. Model terbaik dipilih berdasarkan F1-Score karena kasus churn membutuhkan keseimbangan antara precision dan recall.
6. Deployment sederhana dapat dilakukan menggunakan Streamlit agar model dapat digunakan untuk memprediksi data pelanggan baru.